In [ ]:
# === KONFIGURACJA 5x2cv (Dietterich 1998, Alpaydin 1999) ===
# Protokół: 5 powtórzeń × 2 podziały (50/50 stratified) × 2 architektury = 20 trenowań
#
# UWAGA: ten notebook służy WYŁĄCZNIE do formalnego testu istotności statystycznej.
# Skalary i metryki raportowane w artykule (średnie, odchylenia, macierze pomyłek,
# per-class, krzywe uczenia, Grad-CAM) pochodzą z istniejącego 5-fold CV.

# Konfiguracja powtórzenia
# ZMIEŃ TO PRZED KAŻDĄ SESJĄ: 0, 1, 2, 3 LUB 4
REP_IDX = 4

# Hiperparametry (identyczne jak w 5-fold CV dla spójności porównań)
N_REPETITIONS = 5
NUM_CLASSES = 15
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

USE_CLASS_WEIGHTS = True

PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 20
PHASE1_LR = 1e-3
PHASE2_LR = 1e-4
PATIENCE_P1 = 3
PATIENCE_P2 = 5

RESNET_UNFREEZE = 20
VGG_UNFREEZE = 8

# Ziarna losowości - jedno na powtórzenie (Dietterich zaleca różne seedy
# żeby zapewnić niezależność powtórzeń)
SEEDS_PER_REP = [42, 1042, 2042, 3042, 4042]
RANDOM_SEED = SEEDS_PER_REP[REP_IDX]

# === KAGGLE API ===
KAGGLE_USERNAME = 'myblipe'
KAGGLE_KEY = 'KGAT_b7231294457ba63f62cbfd02db4159b6'

# Ścieżki na Drive
DRIVE_BASE = '/content/drive/MyDrive/plant_disease_classification'
RESULTS_5X2_DIR = f'{DRIVE_BASE}/results/5x2cv'
REP_DIR = f'{RESULTS_5X2_DIR}/rep_{REP_IDX}'

# Lokalna ścieżka do datasetu
LOCAL_DATA = '/content/plantvillage'
SRC = f'{LOCAL_DATA}/PlantVillage'

# Wagi nie są potrzebne (test używa tylko metryk skalarnych)
SAVE_WEIGHTS = False


In [ ]:
# === MONTOWANIE DRIVE ===
import os
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(REP_DIR, exist_ok=True)
os.makedirs(f'{REP_DIR}/split_a', exist_ok=True)
os.makedirs(f'{REP_DIR}/split_b', exist_ok=True)
print(f'Katalog powtórzenia: {REP_DIR}')


In [ ]:
# == KAGGLE ==
import os, json, subprocess

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Pobieranie datasetu jeśli nie ma lokalnie
if not os.path.exists(SRC):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'emmarex/plantdisease',
        '-p', LOCAL_DATA,
        '--unzip',
    ], check=True)
    print(f'Pobrano dataset do {LOCAL_DATA}')
else:
    print(f'Dataset już istnieje w {SRC}')


In [ ]:
# === IMPORTY ===
import time
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix,
)
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, BatchNormalization, Dense, Dropout,
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Globalne ziarno losowości dla całego powtórzenia
tf.keras.utils.set_random_seed(RANDOM_SEED)

print(f'TF: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Powtórzenie {REP_IDX+1}/{N_REPETITIONS}, seed={RANDOM_SEED}')


In [ ]:
# === SPRAWDZENIE STANU POWTÓRZENIA ===
def is_done(split_name, model_name):
    path = f'{REP_DIR}/{split_name}/{model_name.lower()}_metrics.json'
    return os.path.exists(path)

print(f'Stan powtórzenia {REP_IDX+1}:')
for split in ['split_a', 'split_b']:
    for model in ['ResNet50', 'VGG16']:
        status = '✓' if is_done(split, model) else '·'
        print(f'  [{status}] {split}/{model}')

done_marker = f'{REP_DIR}/done.marker'
if os.path.exists(done_marker):
    print('\nPowtórzenie ukończone w całości.')


In [ ]:
# === BUDOWA INDEKSU PLIKÓW (deterministyczny) ===
records = []
for cls in sorted(os.listdir(SRC)):
    cls_path = os.path.join(SRC, cls)
    if not os.path.isdir(cls_path):
        continue
    for f in sorted(os.listdir(cls_path)):
        records.append({'filepath': os.path.join(cls_path, f), 'class': cls})

df = pd.DataFrame(records)
print(f'Łącznie obrazów: {len(df)}')
print(f'Liczba klas: {df["class"].nunique()}')
assert df['class'].nunique() == NUM_CLASSES


In [ ]:
# === STRATIFIED 50/50 SPLIT (z różnym seedem dla każdego powtórzenia) ===
# Dietterich (1998): 5x2cv to 5 powtórzeń 2-fold CV z różnymi losowymi podziałami.
# W każdym powtórzeniu: dane dzielone 50/50 z zachowaniem proporcji klas.
# Każdy split jest niezależnie używany do trenowania (na 50%) i testowania (na drugich 50%).
# Następnie role są odwracane (split_a -> train, split_b -> test; potem odwrotnie).

# Wewnętrzny podział train/val w każdym splicie 50% (z którego trenujemy):
# bierzemy 89% na trening, 11% na walidację (analogicznie do 5-fold CV).
split_a_idx, split_b_idx = train_test_split(
    df.index.values,
    test_size=0.5,
    stratify=df['class'],
    random_state=RANDOM_SEED,
)

print(f'Powtórzenie {REP_IDX+1} (seed={RANDOM_SEED}):')
print(f'  Split A: {len(split_a_idx):>6} obrazów ({len(split_a_idx)/len(df)*100:.1f}%)')
print(f'  Split B: {len(split_b_idx):>6} obrazów ({len(split_b_idx)/len(df)*100:.1f}%)')

# Zapisz indeksy podziału
np.savez(f'{REP_DIR}/split_indices.npz',
         split_a=split_a_idx, split_b=split_b_idx)


In [ ]:
# === FUNKCJE POMOCNICZE: WAGI KLAS + GENERATORY ===
def compute_weights(df_train_subset):
    class_names_sorted = sorted(df_train_subset['class'].unique())
    class_to_idx = {c: i for i, c in enumerate(class_names_sorted)}
    y_train_int = df_train_subset['class'].map(class_to_idx).values

    if USE_CLASS_WEIGHTS:
        weights = compute_class_weight(
            'balanced',
            classes=np.arange(NUM_CLASSES),
            y=y_train_int,
        )
        return dict(enumerate(weights)), class_names_sorted
    return None, class_names_sorted


def make_generators(df_tr, df_va, df_te, preprocess_fn, batch_size=BATCH_SIZE):
    train_gen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
    ).flow_from_dataframe(
        df_tr, x_col='filepath', y_col='class',
        target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=True, seed=RANDOM_SEED,
    )
    val_gen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
    ).flow_from_dataframe(
        df_va, x_col='filepath', y_col='class',
        target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=False,
    )
    test_gen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
    ).flow_from_dataframe(
        df_te, x_col='filepath', y_col='class',
        target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=False,
    )
    return train_gen, val_gen, test_gen


In [ ]:
# === FUNKCJA TRENOWANIA (identyczna logika jak w 5-fold CV) ===
def build_and_train(model_name, base_class, preprocess_fn, n_unfreeze,
                     df_tr, df_va, df_te, class_weights_dict,
                     out_dir):
    print(f'\n{"="*60}\n  {model_name}: powtórzenie {REP_IDX+1}, {os.path.basename(out_dir)}\n{"="*60}')
    t_start = time.time()

    train_gen, val_gen, test_gen = make_generators(df_tr, df_va, df_te, preprocess_fn)

    base = base_class(weights='imagenet', include_top=False,
                      input_shape=(*IMG_SIZE, 3))
    base.trainable = False

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax'),
    ])

    # FAZA 1
    model.compile(
        optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    h1 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=PHASE1_EPOCHS, verbose=2,
        class_weight=class_weights_dict,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=PATIENCE_P1,
                          restore_best_weights=True, verbose=1),
        ],
    )

    # FAZA 2
    base.trainable = False
    n_bn_skipped = 0
    n_unfrozen = 0
    for layer in base.layers[-n_unfreeze:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            n_bn_skipped += 1
        else:
            layer.trainable = True
            n_unfrozen += 1
    print(f'Odblokowano {n_unfrozen} warstw, pominięto {n_bn_skipped} warstw BN')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=PATIENCE_P2,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1),
    ]

    h2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=PHASE2_EPOCHS, verbose=2,
        class_weight=class_weights_dict,
        callbacks=callbacks,
    )

    # OCENA NA ZBIORZE TESTOWYM
    test_gen.reset()
    test_loss, test_acc = model.evaluate(test_gen, verbose=1)

    test_gen.reset()
    y_proba = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(y_proba, axis=1)
    y_true = test_gen.classes

    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')

    elapsed = (time.time() - t_start) / 60
    print(f'\n{model_name}: accuracy={test_acc:.4f}, BA={bal_acc:.4f}, '
          f'F1m={f1_macro:.4f} (czas: {elapsed:.1f} min)')

    results = {
        'model': model_name,
        'repetition': REP_IDX,
        'seed': RANDOM_SEED,
        'split_role': os.path.basename(out_dir),  # 'split_a' lub 'split_b'
        'test_loss': float(test_loss),
        'test_accuracy': float(test_acc),
        'balanced_accuracy': float(bal_acc),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'best_val_acc_phase1': float(max(h1.history['val_accuracy'])),
        'best_val_acc_phase2': float(max(h2.history['val_accuracy'])),
        'epochs_phase1': len(h1.history['val_accuracy']),
        'epochs_phase2': len(h2.history['val_accuracy']),
        'training_time_min': float(elapsed),
        'n_train': len(df_tr),
        'n_val': len(df_va),
        'n_test': len(df_te),
    }

    with open(f'{out_dir}/{model_name.lower()}_metrics.json', 'w') as f:
        json.dump(results, f, indent=2)

    # Zwolnij pamięć
    del model, base
    tf.keras.backend.clear_session()
    gc.collect()

    return results


In [ ]:
# === ROUND 1: trenuj na split_a, testuj na split_b ===
out_dir_1 = f'{REP_DIR}/split_a'

if os.path.exists(f'{out_dir_1}/resnet50_metrics.json') and \
   os.path.exists(f'{out_dir_1}/vgg16_metrics.json'):
    print('Round 1 ukończony, pomijam.')
else:
    # Dla treningu używamy split_a (50%), do testu split_b (50%)
    # Wewnętrznie split_a dzielimy 89%/11% na trening/walidację
    train_idx_r1, val_idx_r1 = train_test_split(
        split_a_idx,
        test_size=0.111,
        stratify=df.iloc[split_a_idx]['class'],
        random_state=RANDOM_SEED,
    )

    df_train_r1 = df.iloc[train_idx_r1].reset_index(drop=True)
    df_val_r1 = df.iloc[val_idx_r1].reset_index(drop=True)
    df_test_r1 = df.iloc[split_b_idx].reset_index(drop=True)

    class_weights_r1, _ = compute_weights(df_train_r1)

    print(f'Round 1: train={len(df_train_r1)}, val={len(df_val_r1)}, test={len(df_test_r1)}')

    # ResNet50
    if not os.path.exists(f'{out_dir_1}/resnet50_metrics.json'):
        build_and_train(
            'ResNet50', ResNet50, preprocess_resnet, RESNET_UNFREEZE,
            df_train_r1, df_val_r1, df_test_r1, class_weights_r1, out_dir_1,
        )

    # VGG16
    if not os.path.exists(f'{out_dir_1}/vgg16_metrics.json'):
        build_and_train(
            'VGG16', VGG16, preprocess_vgg, VGG_UNFREEZE,
            df_train_r1, df_val_r1, df_test_r1, class_weights_r1, out_dir_1,
        )


In [ ]:
# === ROUND 2: trenuj na split_b, testuj na split_a (role odwrócone) ===
out_dir_2 = f'{REP_DIR}/split_b'

if os.path.exists(f'{out_dir_2}/resnet50_metrics.json') and \
   os.path.exists(f'{out_dir_2}/vgg16_metrics.json'):
    print('Round 2 ukończony, pomijam.')
else:
    train_idx_r2, val_idx_r2 = train_test_split(
        split_b_idx,
        test_size=0.111,
        stratify=df.iloc[split_b_idx]['class'],
        random_state=RANDOM_SEED + 1,  # inny seed niż round 1
    )

    df_train_r2 = df.iloc[train_idx_r2].reset_index(drop=True)
    df_val_r2 = df.iloc[val_idx_r2].reset_index(drop=True)
    df_test_r2 = df.iloc[split_a_idx].reset_index(drop=True)

    class_weights_r2, _ = compute_weights(df_train_r2)

    print(f'Round 2: train={len(df_train_r2)}, val={len(df_val_r2)}, test={len(df_test_r2)}')

    if not os.path.exists(f'{out_dir_2}/resnet50_metrics.json'):
        build_and_train(
            'ResNet50', ResNet50, preprocess_resnet, RESNET_UNFREEZE,
            df_train_r2, df_val_r2, df_test_r2, class_weights_r2, out_dir_2,
        )

    if not os.path.exists(f'{out_dir_2}/vgg16_metrics.json'):
        build_and_train(
            'VGG16', VGG16, preprocess_vgg, VGG_UNFREEZE,
            df_train_r2, df_val_r2, df_test_r2, class_weights_r2, out_dir_2,
        )

# Marker ukończenia powtórzenia
with open(done_marker, 'w') as f:
    f.write(f'rep_{REP_IDX} done at {time.ctime()}')
print(f'\n=== Powtórzenie {REP_IDX+1} ukończone ===')


In [ ]:
# === PODSUMOWANIE POWTÓRZENIA ===
print(f'\nWyniki powtórzenia {REP_IDX+1} (seed={RANDOM_SEED}):')
print('-' * 70)
print(f'{"Model":<10} {"Split":<10} {"Accuracy":<10} {"BA":<10} {"F1 macro":<10}')
print('-' * 70)
for split in ['split_a', 'split_b']:
    for model in ['resnet50', 'vgg16']:
        path = f'{REP_DIR}/{split}/{model}_metrics.json'
        if os.path.exists(path):
            with open(path) as f:
                r = json.load(f)
            print(f'{r["model"]:<10} {split:<10} '
                  f'{r["test_accuracy"]*100:.2f}%   '
                  f'{r["balanced_accuracy"]*100:.2f}%   '
                  f'{r["f1_macro"]*100:.2f}%')

print('-' * 70)
print(f'\nGdy ukończysz wszystkie 5 powtórzeń (REP_IDX = 0..4),')
print(f'uruchom zaktualizowany aggregate_colab.ipynb dla testu F-Alpaydina.')
